# 03: Exploratory Data Analysis (EDA) - Customer Churn Analysis
**Author:** Mohamed (Data Analyst)  
**Objective:** Analyze customer behavior metrics to discover key drivers of churn and provide insights for the machine learning model and dashboard.

In [ ]:
import os
import sys

# Tell PySpark exactly where your Java 8 installation live
# Note: Check your computer to see if your path looks exactly like this, or adjust it
os.environ["JAVA_HOME"] = r"C:\Program Files\Eclipse Adoptium\jdk-8.0.492.09-hotspot" 

# Add the bin directory to path just to be completely sure
os.environ["PATH"] = os.environ["JAVA_HOME"] + r"\bin;" + os.environ["PATH"]

# Now load Spark safely
from pyspark.sql import SparkSession
import pyspark.sql.functions as F

spark = SparkSession.builder \
    .appName("Telecom_Churn_EDA") \
    .getOrCreate()

df = spark.read.csv("cleanedData.csv", header=True, inferSchema=True)
print("Data loaded successfully! Row count:", df.count())

## Section 1: Global KPIs & Baseline Churn Rate
Calculating the base rate of how many customers are leaving the company vs. staying.

In [ ]:
# Get total count and group by Churn status
total_customers = df.count()
churn_counts = df.groupBy("Churn").count()

# Calculate the exact percentage
churn_summary = churn_counts.withColumn(
    "Percentage", 
    F.round((F.col("count") / total_customers) * 100, 2)
)

print(f"Total Customers in Dataset: {total_customers}")
churn_summary.show()

## Section 2: Categorical Feature Analysis
Analyzing contract types, internet services, and payment structures to find customer segmentation trends.

In [ ]:
# 1. Aggregate contract data in PySpark
contract_churn = df.groupBy("Contract", "Churn").count()
pdf_contract = contract_churn.toPandas()

# 2. Plotting
plt.figure(figsize=(8, 5))
sns.barplot(data=pdf_contract, x="Contract", y="count", hue="Churn")
plt.title("Customer Churn Count by Contract Type")
plt.xlabel("Contract Type")
plt.ylabel("Number of Customers")
plt.savefig("churn_by_contract.png")
plt.show()

# 1. Aggregate Internet Service data
internet_churn = df.groupBy("InternetService", "Churn").count()
pdf_internet = internet_churn.toPandas()

# 2. Plotting
plt.figure(figsize=(8, 5))
sns.barplot(data=pdf_internet, x="InternetService", y="count", hue="Churn")
plt.title("Customer Churn Count by Internet Service")
plt.xlabel("Internet Service Type")
plt.ylabel("Number of Customers")
plt.savefig("churn_by_internet.png")
plt.show()

In [ ]:
# 1. Aggregate Payment Method data
payment_churn = df.groupBy("PaymentMethod", "Churn").count()
pdf_payment = payment_churn.toPandas()

# 2. Plotting
plt.figure(figsize=(12, 5))
sns.barplot(data=pdf_payment, x="PaymentMethod", y="count", hue="Churn")
plt.title("Customer Churn Count by Payment Method")
plt.xlabel("Payment Method")
plt.ylabel("Number of Customers")
plt.xticks(rotation=15) # Rotates labels slightly so they don't overlap
plt.savefig("churn_by_payment.png")
plt.show()

## Section 3: Financial & Loyalty Metrics (Numerical Analysis)
Examining how long customers stay (tenure) and how much they pay monthly (MonthlyCharges) impacts their retention rate.

In [ ]:
# Calculate average charges and tenure for loyal vs churned customers
financial_summary = df.groupBy("Churn").agg(
    F.round(F.avg("MonthlyCharges"), 2).alias("Avg_Monthly_Bill"),
    F.round(F.avg("tenure"), 2).alias("Avg_Months_With_Company")
)

financial_summary.show()